## Selecting Catalog & Schema

In [0]:
spark.sql('use catalog myproject')

In [0]:
spark.sql('USE SCHEMA bronze')

### Importing Functions from Library

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Checking null counts

In [0]:
df = spark.table("dim_customers_bronze")

# This dynamically generates a count of nulls for every column
null_counts = [sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]

# Apply the calculations and display the summary
df.select(*null_counts).display()

In [0]:
filtered_df = df.filter(df['birthdate'].isNull())
filtered_df.display()

In [0]:
df = spark.table("dim_products_bronze")

# This dynamically generates a count of nulls for every column
null_counts = [sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]

# Apply the calculations and display the summary
df.select(*null_counts).display()

In [0]:
filtered_df = df.filter(df['category'].isNull())
filtered_df.display()

In [0]:
filtered_df = df.filter(df['category_id']=='CO_PE')
filtered_df.display()

In [0]:
df = spark.table("fact_sales_bronze")

# This dynamically generates a count of nulls for every column
null_counts = [sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]

# Apply the calculations and display the summary
df.select(*null_counts).display()

In [0]:
filtered_df = df.filter(df['order_date'].isNull())
filtered_df.display()

## Dropping null rows.

In [0]:
dim_cust=spark.table('dim_customers_bronze')
dim_prod=spark.table('dim_products_bronze')
fact_sales=spark.table('fact_sales_bronze')

In [0]:
tables = ['dim_cust', 'dim_prod', 'fact_sales']

In [0]:
dim_cust.count()

In [0]:
dim_cust=dim_cust.dropna()
dim_prod=dim_prod.dropna()
fact_sales=fact_sales.dropna()

## Changing data type

In [0]:
dim_cust.printSchema()
dim_prod.printSchema()
fact_sales.printSchema()


In [0]:
dim_cust=dim_cust.withColumn('customer_key', col('customer_key').cast('string')) \
.withColumn('customer_id', col('customer_id').cast('string'))

In [0]:
dim_prod=dim_prod.withColumn('product_key', col('product_key').cast('string')) \
.withColumn('product_id', col('product_id').cast('string'))

In [0]:
fact_sales=fact_sales.withColumn('customer_key', col('customer_key').cast('string')) \
.withColumn('product_key', col('product_key').cast('string'))

In [0]:
#verifying datatypes
dim_cust.printSchema()
dim_prod.printSchema()
fact_sales.printSchema()

In [0]:
dim_cust=dim_cust.withColumns({c.name: trim(col(c.name)) for c in dim_cust.schema if c.dataType == StringType()})

In [0]:
dim_prod=dim_prod.withColumns({c.name: trim(col(c.name)) for c in dim_prod.schema if c.dataType == StringType()})

In [0]:
fact_sales=fact_sales.withColumns({c.name: trim(col(c.name)) for c in fact_sales.schema if c.dataType == StringType()})

## dropping duplicates

In [0]:
count = dim_cust.count()
dup_count = dim_cust.dropDuplicates().count()
unique_records = count - dup_count
print(f"Total number of records in the table: {count}\n"
      f"Total number of unique records in the table: {dup_count}\n"
      f"Number of duplicate records: {unique_records}")

In [0]:
count = dim_prod.count()
dup_count = dim_prod.dropDuplicates().count()
unique_records = count - dup_count
print(f"Total number of records in the table: {count}\n"
      f"Total number of unique records in the table: {dup_count}\n"
      f"Number of duplicate records: {unique_records}")

In [0]:
count = fact_sales.count()
dup_count = fact_sales.dropDuplicates().count()
unique_records = count - dup_count
print(f"Total number of records in the table: {count}\n"
      f"Total number of unique records in the table: {dup_count}\n"
      f"Number of duplicate records: {unique_records}")

In [0]:
dim_cust=dim_cust.dropDuplicates()
dim_prod=dim_prod.dropDuplicates()
fact_sales=fact_sales.dropDuplicates()

In [0]:
dim_cust.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("myproject.silver.dim_customers_silver")

dim_prod.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("myproject.silver.dim_products_silver")

fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("myproject.silver.fact_sales_silver")